In [7]:
from gcn import execute, model, data, device

In [15]:
print(data.x)
print(data.x.shape)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
torch.Size([2708, 1433])


In [1]:
from chemXAI import Shap
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset

/home/jonas/Documents/ChemXAI/BranchGNN/GNN/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Supondo que 'data.x' contém as features dos nós (tensor de shape (N, D))
features = data.x  # Tensor (N, D), onde N é o número de nós e D o número de features

# Criando um Dataset PyTorch a partir das features
dataset = TensorDataset(features)

# Definindo tamanhos de treino e teste
train_size = int(0.75 * len(dataset))
test_size = len(dataset) - train_size  # Garante que a soma seja igual ao total

# Dividindo os dados corretamente
train_split, test_split = random_split(dataset, [train_size, test_size])

# Pegando os dados do train_split para verificar o shape
train_data = torch.stack([x[0] for x in train_split])  # Converte lista de tensores em tensor único
test_data = torch.stack([x[0] for x in test_split])  # Converte lista de tensores em tensor único

print(f"Train size: {train_size}\nTest size: {test_size}\n")
print(f"Train split shape: {train_data.shape}\nTest split shape: {test_data.shape}")  # Deve ser (2031, D)

Train size: 2031
Test size: 677

Train split shape: torch.Size([2031, 1433])
Test split shape: torch.Size([677, 1433])


In [11]:
train_loader = DataLoader(train_split, batch_size=16, shuffle=True)
test_loader = DataLoader(test_split, batch_size=16, shuffle=True)

# Pegando um batch do DataLoader
batch_train = next(iter(train_loader))  # Pega o primeiro batch
batch_test = next(iter(test_loader))  # Pega o primeiro batch
# Se os dados forem um TensorDataset, ele retorna (features, labels)
print(f"Train Batch shape (features): {batch_train[0].shape}")  # Shape dos inputs
print(f"Test Batch shape (features): {batch_test[0].shape}")  # Shape dos inputs

# Obtendo `edge_index` do dataset
edge_index = data.edge_index.to(device)
edge_attr = data.edge_attr.to(device) if data.edge_attr is not None else None

# Criando a explicação SHAP
explanation = Shap(model=model, train_loader=train_loader, test_loader=test_loader, device=device, edge_index=edge_index, edge_attr=edge_attr)


Train Batch shape (features): torch.Size([16, 1433])
Test Batch shape (features): torch.Size([16, 1433])
Background shape: (16, 1433)
Test data shape: (16, 1433)


100%|██████████| 16/16 [00:55<00:00,  3.48s/it]


In [12]:
all, _ = explanation.global_explanation(0)
print(all)

Global SHAP values shape: (1433,)
    Feature 0  Feature 1  Feature 2  Feature 3  Feature 4  Feature 5  \
0   -0.000259        0.0        0.0        0.0  -0.000260        0.0   
1    0.002036        0.0        0.0        0.0   0.000000        0.0   
2    0.000502        0.0        0.0        0.0   0.000299        0.0   
3    0.000000        0.0        0.0        0.0   0.000000        0.0   
4    0.000171        0.0        0.0        0.0   0.000000        0.0   
5    0.000000        0.0        0.0        0.0   0.000000        0.0   
6    0.000179        0.0        0.0        0.0   0.000316        0.0   
7    0.000562        0.0        0.0        0.0   0.000000        0.0   
8    0.000309        0.0        0.0        0.0   0.000000        0.0   
9    0.000000        0.0        0.0        0.0  -0.000215        0.0   
10   0.000000        0.0        0.0        0.0   0.000705        0.0   
11   0.000000        0.0        0.0        0.0  -0.000411        0.0   
12   0.000000        0.0      